# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *Magic: The Gathering Comprehensive Rules* (effective 7 June 2024), 292 pages.
**Corpus:** 1,937 rule-aware chunks, shared by both pipelines.

This notebook implements and compares two retrieval pipelines over **exactly the same chunk
corpus, the same top-10 candidate list, and the same prompt template**:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-ranking) |
|---|---|---|
| Stage 1 | Bi-encoder embeds query, FAISS cosine search → top 10 | *identical* |
| Stage 2 | — | Cross-encoder scores all 10 `(query, chunk)` pairs jointly |
| Selection | first 3 of the top 10 | top 3 after re-sorting by cross-encoder score |
| Generation | shared `generate_answer()` | shared `generate_answer()` |

The **only** variable between the two answers is the ordering/selection of the final 3 chunks.

### Why this document
The Comprehensive Rules are ideal for stress-testing retrieval:
- numbered rules with explicit sub-rules (`701.17a`), so gold chunks can be annotated precisely;
- dense with prohibitions ("can't", "may not"), exceptions ("except", "unless"), and penalties;
- heavy repeated terminology — the word *counter* means both "counter a spell" and "a counter
  on a permanent", which creates natural keyword traps.

## Section 0 — Setup

Install the dependencies (uncomment on a fresh machine / Kaggle):

```bash
# !pip install -q sentence-transformers faiss-cpu pypdf pandas numpy
```

The LLM is called through **OpenRouter** (an OpenAI-compatible gateway). The API key is read
from an environment variable so it is never stored in the notebook or the git history:

```bash
export OPENROUTER_API_KEY="sk-or-..."
```

In [1]:
import json
import os
import re
import time
import urllib.error
import urllib.request
from collections import Counter

import faiss
import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer

# ---- Configuration: fixed once, reused by every section -------------------
CONFIG = {
    "pdf_path": "MagicCompRules.pdf",
    "chunks_path": "chunks.json",
    "bi_encoder": "sentence-transformers/all-MiniLM-L6-v2",
    "cross_encoder": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "llm_model": "google/gemini-2.5-flash",
    "top_k": 10,          # candidates retrieved by the bi-encoder
    "top_n": 3,           # chunks handed to the LLM
    "chunk_target": 700,  # soft target chunk size in characters
    "chunk_max": 1100,    # hard cap (cross-encoder truncates at 512 tokens)
    "overlap_rules": 1,   # carry one rule forward between chunks
    "seed": 42,
}

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("torch:", torch.__version__)
print("faiss:", faiss.__version__)

/home/ybk/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu
torch: 2.10.0+cu128
faiss: 1.13.2


## Section 1 — Load and chunk the PDF

### 1.1 Extraction

`pypdf` gives us text per page. We keep the page number as metadata so every chunk can be
traced back to the document.

### 1.2 Cleaning

PDF extraction introduces three artefacts we must repair before chunking:

1. **Hyphenated line breaks** — `"counter-\ned"` → `"countered"`.
2. **Wrapped cross-references** — a line ending in `"See rule"` followed by `"704 ..."` is one
   sentence, not a new rule. Without this fix, `704` becomes a bogus rule number.
3. **Table-of-contents pages** — pages 2–4 are a contents listing. They contain hundreds of rule
   numbers with no rule text, so they would pollute retrieval. We detect and drop them.

### 1.3 Rule-aware chunking

Rather than splitting on raw character counts, we split on **rule boundaries**. Each chunk is a
contiguous run of numbered rules, packed up to ~700 characters, with one rule of overlap so a
rule that straddles a boundary is still fully present in at least one chunk.

This matters for the assignment: a chunk that contains a complete rule (including its
"except when…" sub-rule) is exactly what the LLM needs to answer correctly.

In [2]:
RULE_START = re.compile(r"^(\d{3}(?:\.\d+[a-z]?)?)\.?\s+(.*)$")


def extract_pages(path):
    """Return [{'page': int, 'text': str}] for every page of the PDF."""
    reader = PdfReader(path)
    return [
        {"page": i + 1, "text": page.extract_text() or ""}
        for i, page in enumerate(reader.pages)
    ]


def is_toc_page(text):
    """Heuristic: a page that is mostly short, rule-numbered lines is a table of contents."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) < 8:
        return False
    numbered = sum(1 for l in lines if re.match(r"^\d{3}(\.\d+)?\.?\s", l))
    short = sum(1 for l in lines if len(l) < 60)
    return numbered / len(lines) > 0.5 and short / len(lines) > 0.6


def clean_line(s):
    """Collapse whitespace and repair a trailing hyphen from PDF line wrapping."""
    s = re.sub(r"(\w)-\s*$", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()


def page_to_rules(text):
    """Split one page into [(rule_number, rule_text), ...].

    A line that starts with a rule number begins a new rule. Everything else is a
    continuation of the current rule. The one exception is a wrapped cross-reference:
    if the previous line ends with 'rule'/'rules', the next line is a continuation even
    if it starts with a number (e.g. '... See rule' / '704 See also rule 903.10.').
    """
    rules = []
    cur_num, cur_lines = None, []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if cur_lines and cur_lines[-1].rstrip().endswith(("rule", "rules")):
            cur_lines.append(line)
            continue
        m = RULE_START.match(line)
        if m:
            if cur_lines:
                rules.append((cur_num, " ".join(cur_lines)))
            cur_num, cur_lines = m.group(1), [m.group(2)]
        else:
            cur_lines.append(line)
    if cur_lines:
        rules.append((cur_num, " ".join(cur_lines)))
    return [(n, clean_line(t)) for n, t in rules if t]


def split_long(text, limit):
    """Split an over-long rule at sentence boundaries, then at commas for long lists."""
    if len(text) <= limit:
        return [text]
    parts, buf = [], ""
    for sent in re.split(r"(?<=[.;])\s+", text):
        if buf and len(buf) + len(sent) + 1 > limit:
            parts.append(buf)
            buf = sent
        else:
            buf = (buf + " " + sent).strip()
    if buf:
        parts.append(buf)

    # Some rules are comma-separated lists (e.g. creature types) with no sentence breaks.
    out = []
    for part in parts:
        if len(part) <= limit:
            out.append(part)
            continue
        buf = ""
        for item in part.split(", "):
            if buf and len(buf) + len(item) + 2 > limit:
                out.append(buf + ",")
                buf = item
            else:
                buf = (buf + ", " + item) if buf else item
        if buf:
            out.append(buf)
    return out


def chunk_rules(pages, target, max_len, overlap_rules):
    """Pack consecutive rules into chunks of ~target chars, capped at max_len."""
    chunks = []
    for p in pages:
        if is_toc_page(p["text"]):
            continue
        buf, buf_rules = [], []
        for num, txt in page_to_rules(p["text"]):
            for piece in split_long(f"{num} {txt}" if num else txt, max_len):
                if buf and sum(len(x) for x in buf) + len(piece) > target:
                    chunks.append({"page": p["page"], "rules": buf_rules, "text": "\n".join(buf)})
                    # Carry the last rule(s) forward as overlap, but never let the
                    # overlap itself push the next chunk past max_len.
                    carry = buf[-overlap_rules:] if overlap_rules else []
                    carry_rules = buf_rules[-overlap_rules:] if overlap_rules else []
                    if sum(len(x) for x in carry) + len(piece) > max_len:
                        carry, carry_rules = [], []
                    buf, buf_rules = carry, carry_rules
                buf.append(piece)
                buf_rules.append(num)
        if buf:
            chunks.append({"page": p["page"], "rules": buf_rules, "text": "\n".join(buf)})

    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks


pages = extract_pages(CONFIG["pdf_path"])
chunks = chunk_rules(pages, CONFIG["chunk_target"], CONFIG["chunk_max"], CONFIG["overlap_rules"])

# Persist so chunk IDs stay stable while we annotate queries.
with open(CONFIG["chunks_path"], "w") as f:
    json.dump(chunks, f, indent=1)

lengths = [len(c["text"]) for c in chunks]
print(f"pages: {len(pages)}")
print(f"chunks: {len(chunks)}")
print(f"avg length: {np.mean(lengths):.0f} chars | max: {max(lengths)} chars")
print(f"chunks over {CONFIG['chunk_max']}: {sum(1 for l in lengths if l > CONFIG['chunk_max'])}")

Ignoring wrong pointing object 787 0 (offset 0)


Ignoring wrong pointing object 19312 0 (offset 0)


pages: 292
chunks: 1937
avg length: 642 chars | max: 1099 chars
chunks over 1100: 0


In [3]:
# Inspect a few chunks to confirm the rule-aware boundaries look right.
for c in chunks[40:42]:
    print(f"--- chunk {c['chunk_id']} | page {c['page']} | rules {c['rules']}")
    print(c["text"][:600])
    print()

--- chunk 40 | page 10 | rules ['104.3i', '104.3j', '104.3k', '104.4', '104.4a']
104.3i In an Emperor game, a team loses the game if its emperor loses the game. (See rule 809.5.)
104.3j In a Commander game, a player who’s been dealt 21 or more combat damage by the same commander over the course of the game loses the game. (This is a state-based action. See rule 704. See also rule 903.10.)
104.3k In a tournament, a player may lose the game as a result of a penalty given by a judge. See rule 100.6.
104.4 There are several ways for the game to be a draw.
104.4a If all the players remaining in a game lose simultaneously, the game is a draw.

--- chunk 41 | page 10 | rules ['104.4a', '104.4b', '104.4c', '104.4d']
104.4a If all the players remaining in a game lose simultaneously, the game is a draw.
104.4b If a game that’s not using the limited range of influence option (including a two-player game) somehow enters a “loop” of mandatory actions, repeating a sequence of events with no way to s

## Section 2 — Build the shared index

Both pipelines start from the **same** index, so it is built once here.

- **Bi-encoder:** `all-MiniLM-L6-v2` encodes each chunk independently into a 384-d vector.
  Because query and chunk never see each other, all chunk vectors can be precomputed offline.
- **Similarity:** vectors are L2-normalised, so inner product equals cosine similarity.
- **Index:** FAISS `IndexFlatIP` — exact search, no approximation, appropriate for ~2k vectors.

Index build time is a **one-off** cost and is reported separately from query latency.

In [4]:
t0 = time.perf_counter()
bi_encoder = SentenceTransformer(CONFIG["bi_encoder"], device=DEVICE)
chunk_embeddings = bi_encoder.encode(
    [c["text"] for c in chunks],
    batch_size=64,
    normalize_embeddings=True,   # so inner product == cosine
    show_progress_bar=True,
    convert_to_numpy=True,
)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype(np.float32))
index_build_seconds = time.perf_counter() - t0

print(f"indexed {index.ntotal} chunks of dim {index.d} in {index_build_seconds:.1f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1128.78it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/31 [00:05<02:34,  5.17s/it]

...

Batches:  94%|█████████▎| 29/31 [01:43<00:04,  2.45s/it]

Batches:  97%|█████████▋| 30/31 [01:44<00:02,  2.14s/it]

Batches: 100%|██████████| 31/31 [01:44<00:00,  1.57s/it]

Batches: 100%|██████████| 31/31 [01:44<00:00,  3.38s/it]

indexed 1937 chunks of dim 384 in 113.3s


## Section 3 — Pipeline 1: baseline retrieval

`retrieve_top_k(query, k)` embeds the query and returns the top-k chunks by cosine similarity.
`baseline_top3(query)` simply takes the first three of those ten.

This is the classic dense-retrieval setup: fast, but the bi-encoder compresses each chunk into a
single vector *before* it ever sees the query, so fine-grained relevance signals are lost.

In [5]:
def retrieve_top_k(query, k=None):
    """Return [(chunk_id, cosine_score), ...] sorted by descending cosine similarity."""
    k = k or CONFIG["top_k"]
    q_vec = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, ids = index.search(q_vec.astype(np.float32), k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def baseline_top3(query):
    """Pipeline 1: take the first 3 of the bi-encoder's top 10."""
    return retrieve_top_k(query)[: CONFIG["top_n"]]


# Smoke test
demo = "Can a player sacrifice a permanent they don't control?"
for rank, (cid, score) in enumerate(retrieve_top_k(demo), 1):
    print(f"#{rank:2d}  cos={score:.3f}  [{cid}] {chunks[cid]['text'][:90]}")

# 1  cos=0.570  [796] If you don’t, you lose 4 life.” A player who controls no creatures can’t choose the sacrif
# 2  cos=0.507  [985] 701.17 Sacrifice
701.17a To sacrifice a permanent, its controller moves it from the battle
# 3  cos=0.492  [1404] 704.5x If a Siege’s controller is also its designated protector, that player chooses an op
# 4  cos=0.482  [1567] 720.5a The controller of another player can use only that player’s resources (cards, mana,
# 5  cos=0.473  [965] controlled by the other player. If, on the other hand, those permanents are controlled by 
# 6  cos=0.453  [958] or damage from a source with deathtouch (see rule 704.5h). If a permanent is put into its 
# 7  cos=0.452  [1566] 720.5a The controller of another player can use only that player’s resources (cards, mana,
# 8  cos=0.450  [1103] 702.16k “Protection from [a player]” is a variant of the protection ability. A permanent o
# 9  cos=0.448  [1859] See rules 509.1b–c. Event Anything that happens in a game. See rule 7

## Section 4 — Pipeline 2: cross-encoder re-ranking

The cross-encoder receives the query and a chunk **concatenated into one sequence**, so every
query token can attend to every chunk token. It outputs a single relevance score per pair.

The trade-off: nothing can be precomputed. Each of the 10 candidates needs its own forward pass
at query time. That is why we only run it on the top 10, not the whole corpus.

`rerank()` returns the original bi-encoder rank alongside the new rank, which is what the
rank-shift table in Section 6 is built from.

In [6]:
cross_encoder = CrossEncoder(CONFIG["cross_encoder"], device=DEVICE)


def rerank(query, candidates):
    """Score every (query, chunk) pair jointly and re-sort by cross-encoder score.

    Returns [{'chunk_id', 'bi_score', 'bi_rank', 'ce_score', 'ce_rank'}, ...]
    sorted by descending cross-encoder score.
    """
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)

    rows = []
    for bi_rank, ((cid, bi_score), ce_score) in enumerate(zip(candidates, ce_scores), start=1):
        rows.append({
            "chunk_id": cid,
            "bi_score": bi_score,
            "bi_rank": bi_rank,
            "ce_score": float(ce_score),
        })
    rows.sort(key=lambda r: -r["ce_score"])          # sort by CROSS-ENCODER score
    for ce_rank, row in enumerate(rows, start=1):
        row["ce_rank"] = ce_rank
    return rows


def reranked_top3(query):
    """Pipeline 2: re-rank the same top 10, then take the first 3."""
    return [(r["chunk_id"], r["ce_score"]) for r in rerank(query, retrieve_top_k(query))[: CONFIG["top_n"]]]


# Smoke test: show the full rank shift for one query
demo = "Can a player sacrifice a permanent they don't control?"
print(f"Query: {demo}\n")
print(f"{'new':>4} {'old':>4} {'cos':>7} {'ce':>8}  chunk")
for r in rerank(demo, retrieve_top_k(demo)):
    print(f"{r['ce_rank']:>4} {r['bi_rank']:>4} {r['bi_score']:>7.3f} {r['ce_score']:>+8.2f}  "
          f"[{r['chunk_id']}] {chunks[r['chunk_id']]['text'][:80]}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  90%|█████████ | 95/105 [00:00<00:00, 949.39it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 929.16it/s]


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Can a player sacrifice a permanent they don't control?

 new  old     cos       ce  chunk


   1    2   0.507    +9.25  [985] 701.17 Sacrifice
701.17a To sacrifice a permanent, its controller moves it from 
   2    1   0.570    +4.89  [796] If you don’t, you lose 4 life.” A player who controls no creatures can’t choose 
   3    8   0.450    +4.77  [1103] 702.16k “Protection from [a player]” is a variant of the protection ability. A p
   4    5   0.473    +4.15  [965] controlled by the other player. If, on the other hand, those permanents are cont
   5    9   0.448    +2.37  [1859] See rules 509.1b–c. Event Anything that happens in a game. See rule 700.1. Evoke
   6   10   0.446    +1.17  [14] 101.4 If multiple players would make choices and/or take actions at the same tim
   7    3   0.492    +0.43  [1404] 704.5x If a Siege’s controller is also its designated protector, that player cho
   8    4   0.482    -1.98  [1567] 720.5a The controller of another player can use only that player’s resources (ca
   9    6   0.453    -3.00  [958] or damage from a source with deathtouch (se

## Section 5 — Generation (one shared function)

Both pipelines call the **same** function with the **same** prompt template. The prompt is
defined once as a module-level constant so it is impossible for the two pipelines to drift.

- `temperature = 0` so any difference between Answer 1 and Answer 2 is attributable to the
  retrieved context, not to sampling.
- The model is instructed to answer **only** from the supplied context and to say so when the
  context is insufficient. This makes omissions and hallucinations visible rather than hidden
  behind the model's own knowledge of Magic.

In [7]:
# The prompt is deliberately phrased to avoid false refusals. An earlier version
# ("If the context does not contain the answer, say so") made the model bail out
# whenever a distractor chunk was present, even when the gold chunk was in the
# context. Asking it to cite the rule number keeps it grounded without giving it
# an easy escape hatch.
PROMPT_TEMPLATE = """Answer the question using the context below. Cite the rule number you used.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, chunk_ids):
    """Render the shared prompt. Identical for both pipelines."""
    context = "\n\n".join(
        f"[Chunk {cid} | page {chunks[cid]['page']}]\n{chunks[cid]['text']}"
        for cid in chunk_ids
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


def generate_answer(question, chunk_ids, model=None, max_retries=3):
    """Call the LLM with the shared prompt. Returns the answer text."""
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. Run: export OPENROUTER_API_KEY='sk-or-...'"
        )

    payload = {
        "model": model or CONFIG["llm_model"],
        "messages": [{"role": "user", "content": build_prompt(question, chunk_ids)}],
        "temperature": 0,
    }
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode(),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
    )
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                data = json.load(resp)
            return data["choices"][0]["message"]["content"].strip()
        except urllib.error.HTTPError as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)   # simple exponential backoff on rate limits
        except urllib.error.URLError:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


def run_both_pipelines(question):
    """Run both pipelines on one query and return a comparison record."""
    candidates = retrieve_top_k(question)                 # shared top-10
    ranked = rerank(question, candidates)                 # shared candidate list

    base_ids = [cid for cid, _ in candidates[: CONFIG["top_n"]]]
    rerank_ids = [r["chunk_id"] for r in ranked[: CONFIG["top_n"]]]

    return {
        "question": question,
        "candidates": candidates,
        "ranked": ranked,
        "baseline_ids": base_ids,
        "reranked_ids": rerank_ids,
        "answer_1": generate_answer(question, base_ids),
        "answer_2": generate_answer(question, rerank_ids),
    }


# Verify the two pipelines really do share the candidate list and the prompt.
# Re-ranking must only REORDER the top-10, never add or drop a candidate.
_q = "Can a player sacrifice a permanent they don't control?"
_c = retrieve_top_k(_q)
_reranked = rerank(_q, _c)

assert len(_c) == CONFIG["top_k"], "baseline did not return exactly top_k candidates"
assert len(_reranked) == len(_c), "re-ranking changed the number of candidates"
assert {cid for cid, _ in _c} == {r["chunk_id"] for r in _reranked}, \
    "re-ranking changed the candidate SET, not just the order"
assert build_prompt(_q, [1, 2]) == build_prompt(_q, [1, 2]), "prompt is not deterministic"

print(f"shared candidate set verified: {len(_c)} chunks, reordered only")
print("shared prompt template verified")

shared candidate set verified: 10 chunks, reordered only
shared prompt template verified


## Section 6 — Failure-mode stress tests

### 6.1 How the queries were found

I did not guess these. The workflow was:

1. Write ~100 candidate questions across the four required categories.
2. For each, locate the **gold chunk(s)** by searching the chunk corpus for the rule number that
   actually answers the question (e.g. `701.17a` for sacrifice).
3. Run the baseline and record the gold chunk's rank in the top 10.
4. Run the re-ranker and record the new rank.
5. Keep the queries where gold moves from **rank 4–10 into rank 1–3**, and keep at least one
   case where re-ranking does **not** help.

### 6.2 The four required categories

| Category | Query | Why it is hard for the bi-encoder |
|---|---|---|
| **Keyword trap** | "Can a player look at another player's hand?" | The word *hand* appears in dozens of unrelated rules (hand size, hand modifier, starting hand). The answer is buried in `402.3`. |
| **Exception / corner case** | "What happens if a player would both win and lose the game at the same time?" | The general win/lose rules dominate; the specific tie-break is one sub-rule (`104.3f`). |
| **Negation / constraint** | "Can you respond to a mana ability?" | The bi-encoder blurs "mana ability" with "respond"; the answer is a *negative* rule (`605.3b`: it can't be responded to). |
| **Buried chunk** | "What happens when a planeswalker has zero loyalty?" | The answer is a state-based action (`704.5i`) buried among dozens of other SBAs. |
### 6.3 Query set

Each entry records the gold rule number(s). The gold chunk IDs are resolved automatically from
the corpus, so the annotation is reproducible rather than hand-copied.

In [8]:
# (query, category, gold rule numbers, note)
TEST_QUERIES = [
    # --- keyword trap / distractor -----------------------------------------
    ("Can a player look at another player's hand?",
     "keyword trap", ["402.3"],
     "The word 'hand' is everywhere; the answer is one sentence in 402.3."),
    ("Can a player target a creature with protection?",
     "keyword trap", ["702.16b"],
     "Baseline surfaces 'target' rules; the protection restriction is elsewhere."),
    ("Can a player tap a permanent that is already tapped?",
     "keyword trap", ["701.21a"],
     "Baseline prefers 'tap for mana' rules; the actual rule is 701.21a."),

    # --- exceptions & corner cases -----------------------------------------
    ("What happens if a player would both win and lose the game at the same time?",
     "exception", ["104.3f"],
     "General win/lose rules dominate; the tie-break is a single sub-rule."),
    ("Can a player activate a mana ability while casting a spell?",
     "exception", ["605.3a"],
     "An exception to normal timing, buried in the mana-ability section."),
    ("What happens when a creature with trample is blocked by a creature with protection?",
     "exception", ["702.19b", "702.16e"],
     "Requires combining the trample assignment rule with the protection rule."),

    # --- negation & constraints --------------------------------------------
    ("Can you respond to a mana ability?",
     "negation", ["605.3b"],
     "The answer is a prohibition: it can't be responded to."),
    ("Can a player sacrifice a permanent that is already in the graveyard?",
     "negation", ["701.17a"],
     "The rule states a player can't sacrifice a non-permanent."),

    # --- buried chunk -------------------------------------------------------
    ("What happens when a planeswalker has zero loyalty?",
     "buried", ["704.5i"],
     "One state-based action among many in rule 704.5."),
    ("What happens when a creature has lethal damage marked on it?",
     "buried", ["704.5g"],
     "Another state-based action buried in the same long list."),

    # --- honest negative control: re-ranking does NOT help ------------------
    ("What happens when a spell is countered?",
     "control (no gain)", ["701.5a"],
     "Baseline already ranks the gold chunk #1; re-ranking pushes it down."),
]


def find_chunks_by_rule(rule):
    """Return chunk IDs whose text contains this rule number as a whole token."""
    pattern = re.compile(r"(?<![\d.])" + re.escape(rule) + r"(?![\d])")
    return [c["chunk_id"] for c in chunks if pattern.search(c["text"])]


def rank_of_gold(ordered_ids, gold_ids):
    """1-based rank of the best-ranked gold chunk, or None if absent."""
    ranks = [ordered_ids.index(g) + 1 for g in gold_ids if g in ordered_ids]
    return min(ranks) if ranks else None


records = []
for query, category, gold_rules, note in TEST_QUERIES:
    gold_ids = sorted({cid for rule in gold_rules for cid in find_chunks_by_rule(rule)})
    candidates = retrieve_top_k(query)
    ranked = rerank(query, candidates)

    bi_ids = [cid for cid, _ in candidates]
    ce_ids = [r["chunk_id"] for r in ranked]

    records.append({
        "query": query,
        "category": category,
        "gold_rules": ", ".join(gold_rules),
        "gold_chunk_ids": gold_ids,
        "baseline_rank": rank_of_gold(bi_ids, gold_ids),
        "reranked_rank": rank_of_gold(ce_ids, gold_ids),
        "note": note,
    })

summary = pd.DataFrame(records)
summary["rank_shift"] = summary.apply(
    lambda r: f"#{r.baseline_rank} -> #{r.reranked_rank}"
    if pd.notna(r.baseline_rank) and pd.notna(r.reranked_rank) else "n/a",
    axis=1,
)
summary["fixed_in_top3"] = summary.apply(
    lambda r: bool(pd.notna(r.baseline_rank) and pd.notna(r.reranked_rank)
                   and r.baseline_rank > CONFIG["top_n"] and r.reranked_rank <= CONFIG["top_n"]),
    axis=1,
)

pd.set_option("display.max_colwidth", 70)
summary[["category", "query", "gold_rules", "baseline_rank", "reranked_rank", "rank_shift", "fixed_in_top3"]]

,category,query,gold_rules,baseline_rank,reranked_rank,rank_shift,fixed_in_top3
0,keyword trap,Can a player look at another player's hand?,402.3,8,1,#8 -> #1,True
1,keyword trap,Can a player target a creature with protection?,702.16b,9,3,#9 -> #3,True
2,keyword trap,Can a player tap a permanent that is already tapped?,701.21a,5,1,#5 -> #1,True
3,exception,What happens if a player would both win and lose the game at the s...,104.3f,7,1,#7 -> #1,True
4,exception,Can a player activate a mana ability while casting a spell?,605.3a,8,2,#8 -> #2,True
5,exception,What happens when a creature with trample is blocked by a creature...,"702.19b, 702.16e",5,2,#5 -> #2,True
6,negation,Can you respond to a mana ability?,605.3b,8,1,#8 -> #1,True
7,negation,Can a player sacrifice a permanent that is already in the graveyard?,701.17a,4,1,#4 -> #1,True
8,buried,What happens when a planeswalker has zero loyalty?,704.5i,8,2,#8 -> #2,True
9,buried,What happens when a creature has lethal damage marked on it?,704.5g,5,2,#5 -> #2,True


In [9]:
# Full top-10 with both scores for every test query, so the shifts are visible.
for rec in records:
    query = rec["query"]
    gold = set(rec["gold_chunk_ids"])
    candidates = retrieve_top_k(query)
    ranked = rerank(query, candidates)

    print("=" * 100)
    print(f"{query}")
    print(f"category: {rec['category']} | gold rules: {rec['gold_rules']} | gold chunks: {sorted(gold)}")
    print(f"baseline rank: {rec['baseline_rank']}  ->  re-ranked rank: {rec['reranked_rank']}")
    print(f"{'new':>4} {'old':>4} {'cos':>7} {'ce':>8}  chunk")
    for r in ranked:
        mark = "  <== GOLD" if r["chunk_id"] in gold else ""
        text = chunks[r["chunk_id"]]["text"][:85].replace("\n", " ")
        print(f"{r['ce_rank']:>4} {r['bi_rank']:>4} {r['bi_score']:>7.3f} {r['ce_score']:>+8.2f}  "
              f"[{r['chunk_id']}] {text}{mark}")
    print()

Can a player look at another player's hand?
category: keyword trap | gold rules: 402.3 | gold chunks: [524]
baseline rank: 8  ->  re-ranked rank: 1
 new  old     cos       ce  chunk
   1    8   0.369    +7.74  [524] 402.2 Each player has a maximum hand size, which is normally seven cards. A player ma  <== GOLD
   2    1   0.496    +1.32  [1564] 720.3 Only control of the player changes. All objects are controlled by their normal 
   3    3   0.408    +1.17  [1502] 712.6 Players who are allowed to look at a double-faced card may look at both sides o
   4    6   0.380    +1.06  [1569] 720.6 The controller of another player can’t make that player concede. A player may c
   5    7   0.373    +0.87  [1568] 720.5b The controller of another player can’t make choices or decisions for that play
   6    4   0.389    +0.74  [1844] Control Another Player To make all choices and decisions that player is allowed to ma
   7    2   0.415    -0.22  [1741] 811.4 A player can’t attack opponents who aren’t

Can a player target a creature with protection?
category: keyword trap | gold rules: 702.16b | gold chunks: [1096, 1097]
baseline rank: 9  ->  re-ranked rank: 3
 new  old     cos       ce  chunk
   1    5   0.579    +6.97  [1103] 702.16k “Protection from [a player]” is a variant of the protection ability. A perman
   2    8   0.521    +6.56  [1104] 702.16k “Protection from [a player]” is a variant of the protection ability. A perman
   3    9   0.515    +6.21  [1097] 702.16b A permanent or player with protection can’t be targeted by spells with the st  <== GOLD
   4    1   0.613    +5.30  [1099] 702.16d A permanent with protection can’t be equipped by Equipment that have the stat
   5    7   0.530    +5.28  [477] 310.8c A battle’s protector may block creatures attacking that battle with creatures 
   6    4   0.581    +5.27  [1102] 702.16i “Protection from each [set of characteristics, qualities, or players]” is sho
   7    6   0.552    +4.81  [476] 310.8a As a battle enters the battle

Can a player tap a permanent that is already tapped?
category: keyword trap | gold rules: 701.21a | gold chunks: [1002]
baseline rank: 5  ->  re-ranked rank: 1
 new  old     cos       ce  chunk
   1    5   0.503    +6.58  [1002] 701.21 Tap and Untap 701.21a To tap a permanent, turn it sideways from an upright pos  <== GOLD
   2    8   0.456    +4.89  [60] 106.12b A replacement effect that applies if a permanent “is tapped for mana” or tapp
   3    2   0.558    +4.00  [716] 603.2f Some trigger events use the word “becomes” (for example, “becomes attached” or
   4    1   0.572    +3.99  [59] 106.12 To “tap [a permanent] for mana” is to activate a mana ability of that permanen
   5    7   0.491    +2.82  [1029] 701.39b A permanent can be exerted even if it’s not tapped or has already been exerte
   6    3   0.537    +0.33  [715] 603.2e Some effects refer to a triggered ability of an object. Such effects refer onl
   7    4   0.511    +0.02  [1485] 710.3 You must ensure that it’s clear at 

What happens if a player would both win and lose the game at the same time?
category: exception | gold rules: 104.3f | gold chunks: [39]
baseline rank: 7  ->  re-ranked rank: 1
 new  old     cos       ce  chunk
   1    7   0.478    +7.87  [39] 104.3e An effect may state that a player loses the game. 104.3f If a player would bot  <== GOLD
   2    5   0.501    +6.17  [1728] 810.8 The Two-Headed Giant variant uses the normal rules for winning or losing the ga
   3    3   0.535    +3.83  [40] 104.3i In an Emperor game, a team loses the game if its emperor loses the game. (See 
   4    1   0.566    +3.45  [1729] Example: In a Two-Headed Giant game, a player controls Platinum Angel, which reads, “
   5    2   0.538    +3.36  [36] 104.2 There are several ways to win the game. 104.2a A player still in the game wins 
   6    4   0.513    +3.00  [37] 104.2d In an Emperor game, a team wins the game if its emperor wins the game. (See ru
   7   10   0.459    +2.60  [1730] 810.8b If a player concede

Can a player activate a mana ability while casting a spell?
category: exception | gold rules: 605.3a | gold chunks: [760]
baseline rank: 8  ->  re-ranked rank: 2
 new  old     cos       ce  chunk
   1    2   0.698   +10.18  [207] 117.1d A player may activate a mana ability whenever they have priority, whenever the
   2    8   0.595    +9.66  [760] 605.3 Activating an activated mana ability follows the rules for activating any other  <== GOLD
   3    9   0.591    +8.64  [762] 605.4a A triggered mana ability doesn’t go on the stack, so it can’t be targeted, cou
   4    3   0.648    +7.33  [764] 605.5a An ability with a target is not a mana ability, even if it could put mana into
   5    5   0.635    +5.83  [763] 605.5 Abilities that don’t meet the criteria specified in rules 605.1a–b and spells a
   6    4   0.647    +5.66  [758] 605.1b A triggered ability is a mana ability if it meets all of the following criteri
   7    7   0.601    +5.45  [54] 106.5 If an ability would produce one or 

...

Can you respond to a mana ability?
category: negation | gold rules: 605.3b | gold chunks: [760, 761]
baseline rank: 8  ->  re-ranked rank: 1
 new  old     cos       ce  chunk
   1    8   0.549    +5.03  [761] 605.3b An activated mana ability doesn’t go on the stack, so it can’t be targeted, co  <== GOLD
   2    4   0.585    +4.59  [762] 605.4a A triggered mana ability doesn’t go on the stack, so it can’t be targeted, cou
   3    2   0.602    +2.85  [207] 117.1d A player may activate a mana ability whenever they have priority, whenever the
   4    1   0.617    +2.48  [759] 605.2 A mana ability remains a mana ability even if the game state doesn’t allow it t
   5    3   0.594    +2.46  [758] 605.1b A triggered ability is a mana ability if it meets all of the following criteri
   6    7   0.563    +2.18  [764] 605.5a An ability with a target is not a mana ability, even if it could put mana into
   7    6   0.563    +1.67  [763] 605.5 Abilities that don’t meet the criteria specified in rul

Can a player sacrifice a permanent that is already in the graveyard?
category: negation | gold rules: 701.17a | gold chunks: [985]
baseline rank: 4  ->  re-ranked rank: 1
 new  old     cos       ce  chunk
   1    4   0.511    +7.41  [985] 701.17 Sacrifice 701.17a To sacrifice a permanent, its controller moves it from the b  <== GOLD
   2    2   0.529    +3.33  [1833] See rule 903.12, “Brawl Option.” Bury (Obsolete) A term that meant “put [a permanent]
   3    1   0.569    +3.00  [958] or damage from a source with deathtouch (see rule 704.5h). If a permanent is put into
   4    5   0.510    +2.33  [528] 404 Graveyard 404.1 A player’s graveyard is their discard pile. Any object that’s cou
   5    8   0.472    +2.23  [810] 608.3e If a permanent spell resolves but its controller can’t put it onto the battlef
   6    3   0.527    +0.36  [957] 701.7a To destroy a permanent, move it from the battlefield to its owner’s graveyard.
   7   10   0.467    -3.40  [1186] 702.48b You choose which perm

What happens when a planeswalker has zero loyalty?
category: buried | gold rules: 704.5i | gold chunks: [1394, 1395]
baseline rank: 8  ->  re-ranked rank: 2
 new  old     cos       ce  chunk
   1    6   0.664    +6.65  [288] 122.1e The number of loyalty counters on a planeswalker on the battlefield indicates 
   2    8   0.655    +6.53  [1395] 704.5i If a planeswalker has loyalty 0, it’s put into its owner’s graveyard. 704.5j I  <== GOLD
   3   10   0.597    +4.13  [404] power and/or toughness, ignoring any effects and counters that modify power and/or to
   4    2   0.751    +3.87  [457] 306.6 Planeswalkers can be attacked. (See rule 508, “Declare Attackers Step.”) 306.7 
   5    1   0.782    +3.06  [456] 306.5c The loyalty of a planeswalker on the battlefield is equal to the number of loy
   6    4   0.718    +2.74  [455] 306.5a The loyalty of a planeswalker card not on the battlefield is equal to the numb
   7    7   0.657    +1.92  [766] 606.2 An activated ability with a loyalty sy

What happens when a creature has lethal damage marked on it?
category: buried | gold rules: 704.5g | gold chunks: [957, 1089, 1393, 1394, 1878]
baseline rank: 5  ->  re-ranked rank: 2
 new  old     cos       ce  chunk
   1    1   0.578    +6.94  [266] 120.6 Damage marked on a creature remains until the cleanup step, even if that perman
   2    5   0.546    +6.88  [1394] 704.5g If a creature has toughness greater than 0, it has damage marked on it, and th  <== GOLD
   3    3   0.570    +5.29  [265] 120.5 Damage dealt to a creature, planeswalker, or battle doesn’t destroy it. Likewis
   4    2   0.577    +5.16  [427] 302.6 A creature’s activated ability with the tap symbol or the untap symbol in its a
   5    7   0.507    +4.79  [659] 510.1d A blocking creature assigns combat damage to the creatures it’s blocking. If i
   6    6   0.524    +4.73  [655] 510.1c A blocked creature assigns its combat damage to the creatures blocking it. If 
   7   10   0.483    +0.59  [264] 120.5 Damage deal

What happens when a spell is countered?
category: control (no gain) | gold rules: 701.5a | gold chunks: [954]
baseline rank: 1  ->  re-ranked rank: 4
 new  old     cos       ce  chunk
   1    7   0.416    +5.44  [746] 603.10e Abilities that trigger when a spell is countered look back in time. 603.10f A
   2    9   0.406    +3.35  [794] 608.2c The controller of the spell or ability follows its instructions in the order w
   3    3   0.522    +2.65  [140] 112 Spells 112.1 A spell is a card on the stack. As the first step of being cast (see
   4    1   0.686    +2.02  [954] 701.4 Cast 701.4a To cast a spell is to take it from the zone it’s in (usually the ha  <== GOLD
   5    2   0.527    +2.01  [675] 601.2a To propose the casting of a spell, a player first moves that card (or that cop
   6    5   0.428    +1.07  [762] 605.4a A triggered mana ability doesn’t go on the stack, so it can’t be targeted, cou
   7   10   0.397    -0.21  [912] 615.8 Some prevention effects generated by the resol

### 6.4 Answer 1 vs. Answer 2

For each test query we now generate both answers with the identical prompt and compare them.
Watch for the three failure signatures the assignment asks about:

- **hallucination** — the model states something not present in the supplied context;
- **omission** — the model gives the general rule but drops the exception/restriction;
- **misstatement** — the model states the rule backwards because the critical chunk was missing.

In [10]:
results = []
for rec in records:
    out = run_both_pipelines(rec["query"])
    results.append({
        "query": rec["query"],
        "category": rec["category"],
        "baseline_ids": out["baseline_ids"],
        "reranked_ids": out["reranked_ids"],
        "answer_1": out["answer_1"],
        "answer_2": out["answer_2"],
    })

for r in results:
    print("=" * 100)
    print(f"QUERY: {r['query']}   [{r['category']}]")
    print(f"\n--- Answer 1 (baseline, chunks {r['baseline_ids']}) ---")
    print(r["answer_1"])
    print(f"\n--- Answer 2 (re-ranked, chunks {r['reranked_ids']}) ---")
    print(r["answer_2"])
    print()

QUERY: Can a player look at another player's hand?   [keyword trap]

--- Answer 1 (baseline, chunks [1564, 1741, 1502]) ---
Yes, a player can look at another player's hand under specific circumstances.

**Rule 720.4** states that if a player is being controlled, the controller of that player can see their hand.

**Rule 811.5** states that in the Alternating Teams variant, teammates can review each other's hands if they are sitting next to each other.

--- Answer 2 (re-ranked, chunks [524, 1564, 1502]) ---
No, a player cannot look at the cards in another player's hand. However, they may count those cards at any time.

Rule used: 402.3

QUERY: Can a player target a creature with protection?   [keyword trap]

--- Answer 1 (baseline, chunks [1099, 1100, 626]) ---
No, a player cannot target a creature with protection. This is covered by rule 702.16e, which states that any damage that would be dealt by sources that have the stated quality to a permanent or player with protection is prevented

## Section 7 — Latency profiling

Method:

- `time.perf_counter()` around each stage.
- **Warm-up** both models with one dummy call first, so first-call overhead (lazy CUDA init,
  tokenizer caching) does not pollute the numbers.
- Repeat every query **15 times** and report **mean ± std**.
- Time the stages separately: query embedding, FAISS search, cross-encoder scoring.
- LLM generation is reported separately because it is dominated by network latency and is
  noisy; it is not part of the retrieval comparison.

In [11]:
def time_stage(fn, repeats=15):
    """Return (mean_ms, std_ms) for fn() over `repeats` runs."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))


# ---- Warm-up: one dummy call per model so first-call cost is excluded ----
_ = bi_encoder.encode(["warm up"], normalize_embeddings=True)
_ = cross_encoder.predict([("warm up", "warm up")])
_ = index.search(np.zeros((1, index.d), dtype=np.float32), CONFIG["top_k"])

latency_rows = []
for rec in records:
    query = rec["query"]
    candidates = retrieve_top_k(query)

    embed_ms, embed_sd = time_stage(
        lambda: bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    )
    q_vec = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    search_ms, search_sd = time_stage(
        lambda: index.search(q_vec, CONFIG["top_k"])
    )
    ce_ms, ce_sd = time_stage(
        lambda: cross_encoder.predict([(query, chunks[cid]["text"]) for cid, _ in candidates],
                                      show_progress_bar=False)
    )

    latency_rows.append({
        "query": query[:45],
        "embed_ms": embed_ms,
        "search_ms": search_ms,
        "baseline_total_ms": embed_ms + search_ms,
        "cross_encoder_ms": ce_ms,
        "rerank_total_ms": embed_ms + search_ms + ce_ms,
    })

lat = pd.DataFrame(latency_rows)
lat.round(2)

,query,embed_ms,search_ms,baseline_total_ms,cross_encoder_ms,rerank_total_ms
0,Can a player look at another player's hand?,40.83,2.17,42.99,593.64,636.63
1,Can a player target a creature with protectio,27.61,0.48,28.10,516.49,544.59
2,Can a player tap a permanent that is already,26.55,1.40,27.95,741.44,769.38
3,What happens if a player would both win and l,32.26,0.32,32.58,721.01,753.59
4,Can a player activate a mana ability while ca,28.24,2.13,30.36,734.11,764.47
5,What happens when a creature with trample is,28.33,1.80,30.13,598.90,629.03
6,Can you respond to a mana ability?,35.68,2.12,37.80,736.82,774.62
7,Can a player sacrifice a permanent that is al,30.54,2.26,32.80,570.91,603.71
8,What happens when a planeswalker has zero loy,42.35,2.17,44.52,627.84,672.37
9,What happens when a creature has lethal damag,25.68,0.40,26.08,682.47,708.55


In [12]:
# Aggregate latency table (mean over all test queries).
agg = pd.DataFrame({
    "stage": [
        "Query embedding (bi-encoder)",
        "FAISS search (top-10)",
        "Baseline retrieval total",
        "Cross-encoder scoring (10 pairs)",
        "Re-ranking pipeline total",
    ],
    "mean_ms": [
        lat["embed_ms"].mean(),
        lat["search_ms"].mean(),
        lat["baseline_total_ms"].mean(),
        lat["cross_encoder_ms"].mean(),
        lat["rerank_total_ms"].mean(),
    ],
    "std_ms": [
        lat["embed_ms"].std(),
        lat["search_ms"].std(),
        lat["baseline_total_ms"].std(),
        lat["cross_encoder_ms"].std(),
        lat["rerank_total_ms"].std(),
    ],
})
agg["mean_ms"] = agg["mean_ms"].round(2)
agg["std_ms"] = agg["std_ms"].round(2)

overhead = lat["cross_encoder_ms"].mean()
baseline = lat["baseline_total_ms"].mean()
print(f"Hardware: {DEVICE.upper()} | torch {torch.__version__}")
print(f"Index build (one-off): {index_build_seconds:.1f}s for {index.ntotal} chunks")
print(f"Re-ranking overhead: {overhead:.2f} ms  ({overhead / baseline:.1f}x the baseline retrieval)")
print(f"Per-pair cross-encoder cost: {overhead / CONFIG['top_k']:.2f} ms")
agg

Hardware: CPU | torch 2.10.0+cu128
Index build (one-off): 113.3s for 1937 chunks
Re-ranking overhead: 635.65 ms  (19.0x the baseline retrieval)
Per-pair cross-encoder cost: 63.56 ms


,stage,mean_ms,std_ms
0,Query embedding (bi-encoder),31.91,5.64
1,FAISS search (top-10),1.60,0.81
2,Baseline retrieval total,33.51,6.09
3,Cross-encoder scoring (10 pairs),635.65,94.93
4,Re-ranking pipeline total,669.15,94.22


## Section 8 — Summary and observations

### 8.1 Rank-shift summary

In [13]:
summary[["category", "query", "gold_rules", "baseline_rank", "reranked_rank", "rank_shift", "fixed_in_top3"]]

,category,query,gold_rules,baseline_rank,reranked_rank,rank_shift,fixed_in_top3
0,keyword trap,Can a player look at another player's hand?,402.3,8,1,#8 -> #1,True
1,keyword trap,Can a player target a creature with protection?,702.16b,9,3,#9 -> #3,True
2,keyword trap,Can a player tap a permanent that is already tapped?,701.21a,5,1,#5 -> #1,True
3,exception,What happens if a player would both win and lose the game at the s...,104.3f,7,1,#7 -> #1,True
4,exception,Can a player activate a mana ability while casting a spell?,605.3a,8,2,#8 -> #2,True
5,exception,What happens when a creature with trample is blocked by a creature...,"702.19b, 702.16e",5,2,#5 -> #2,True
6,negation,Can you respond to a mana ability?,605.3b,8,1,#8 -> #1,True
7,negation,Can a player sacrifice a permanent that is already in the graveyard?,701.17a,4,1,#4 -> #1,True
8,buried,What happens when a planeswalker has zero loyalty?,704.5i,8,2,#8 -> #2,True
9,buried,What happens when a creature has lethal damage marked on it?,704.5g,5,2,#5 -> #2,True


In [14]:
fixed = summary[summary["fixed_in_top3"]]
print(f"Queries where re-ranking moved gold from rank 4-10 into the top 3: {len(fixed)} / {len(summary)}")
print()
for _, r in fixed.iterrows():
    print(f"  [{r['category']}] {r['query']}")
    print(f"      gold {r['gold_rules']} : {r['rank_shift']}")

Queries where re-ranking moved gold from rank 4-10 into the top 3: 10 / 11

  [keyword trap] Can a player look at another player's hand?
      gold 402.3 : #8 -> #1
  [keyword trap] Can a player target a creature with protection?
      gold 702.16b : #9 -> #3
  [keyword trap] Can a player tap a permanent that is already tapped?
      gold 701.21a : #5 -> #1
  [exception] What happens if a player would both win and lose the game at the same time?
      gold 104.3f : #7 -> #1
  [exception] Can a player activate a mana ability while casting a spell?
      gold 605.3a : #8 -> #2
  [exception] What happens when a creature with trample is blocked by a creature with protection?
      gold 702.19b, 702.16e : #5 -> #2
  [negation] Can you respond to a mana ability?
      gold 605.3b : #8 -> #1
  [negation] Can a player sacrifice a permanent that is already in the graveyard?
      gold 701.17a : #4 -> #1
  [buried] What happens when a planeswalker has zero loyalty?
      gold 704.5i : #8 -> #2
 

### 8.2 What the results show

**Where re-ranking helps.** The gains cluster in exactly the cases the assignment predicts.
When the query shares surface vocabulary with an irrelevant passage — *hand*, *tap*, *target*,
*mana ability* — the bi-encoder's single pooled vector is pulled toward the lexical match. The
cross-encoder sees the query and chunk together, so it can tell that "look at another player's
hand" is about *information access* while the top-ranked baseline chunk is about *hand size*.

The clearest answer-level failures are:

- **Misstatement.** "Would a player who both wins and loses simultaneously win or lose?" —
  Answer 1 said *the game is a draw* (rule 104.4a) because the tie-break rule 104.3f sat at
  rank 7. Answer 2 states the correct outcome: the player **loses**.
- **Omission.** "Can a player look at another player's hand?" — Answer 1 inferred a hedged
  answer from adjacent rules (811.5, 720.4) instead of retrieving 402.3. Answer 2 quotes the
  actual prohibition.
- **Refusal.** "Can you respond to a mana ability?" and the trample/protection query both
  produced "the context does not contain the answer" under the baseline, and correct answers
  after re-ranking.

**Where it does not help.** The negative control ("What happens when a spell is countered?")
starts with the gold chunk at rank 1 and re-ranking pushes it to rank 4. The cross-encoder
prefers chunks that *mention* countering in passing over the chunk that *defines* it. This is a
real limitation: a cross-encoder trained on MS MARCO is optimised for web-passage relevance, not
for "find the definitional rule", and it can demote a correct top-1 result.

**Cost.** Re-ranking adds a fixed cost proportional to `k` (10 forward passes), not to corpus
size. That is the whole point of retrieve-then-rerank: the expensive model only ever sees a
handful of candidates.

### 8.3 Scaling

A cross-encoder cannot precompute anything: the score depends on the *pair*, so every
`(query, chunk)` combination needs its own forward pass. Running it over the full corpus would
cost `N` forward passes per query. With the measured per-pair cost above, a 1,937-chunk corpus
would take roughly `1937 × per-pair cost` per query — and a 1M-chunk corpus would be ~500×
worse again, with no way to cache the result because the query changes every time.

The bi-encoder, by contrast, embeds the corpus **once** and then answers each query with a
single embedding plus a vector lookup. That asymmetry is why the two-stage design exists:
use the cheap model to cut `N` down to `k`, then spend the expensive model's budget only on
those `k` candidates.

## Section 9 — Reproducibility checklist

- [x] One chunk corpus, saved to `chunks.json`, used by both pipelines.
- [x] Both pipelines consume the **same** `retrieve_top_k()` output.
- [x] One `PROMPT_TEMPLATE` constant, used by both pipelines.
- [x] `temperature = 0`.
- [x] Gold chunks annotated by rule number, resolved programmatically.
- [x] At least two baseline-fail → re-rank-fix queries, plus all four categories.
- [x] One honest non-improving case.
- [x] Latency measured with warm-up and repeated runs.
- [x] API key read from the environment, never stored in the notebook.